In [1]:
import matplotlib.pyplot as plt
import datetime
import numpy as np
import time

In [2]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import site_archive_jasmin # Required to not get missing something error

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/ssde/j25a/mmh_storage/theme3/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25a/mmh_storage/theme3/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25a/mmh_storage/theme3/ra22_himawari', 'Rainfields3': '/gws/ssde/j25a/mmh_storage/theme3/rq0_rainfields_prcp_crate'}


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
selected_date = datetime.datetime(2021,6,9,2,0)
himawari = petdata.archive.Himawari('surface_global_irradiance')
rf3proj = petdata.transforms.projection.Rainfields3ProjAus()
radar_projector = petdata.transforms.projection.XYtoLonLatRectilinear(rf3proj)
satpipe = petpipe.Pipeline(
    himawari
)


In [6]:
fullsat = petpipe.Pipeline(
    satpipe,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20200101T00', '20210101T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

In [8]:
# Reminder, the image size is latitude: 1726, longitude: 2214

class AutoEncoder(nn.Module):
    def __init__(self, 
                 input_height = 501,
                 input_width = 601,
                 kernel_size = 4,
                 stride=2,
                 input_channel_count = 2,
                 output_channel_count = 2,
                 latent_dim=300):
        super(AutoEncoder, self).__init__()

        self.input_width = input_width
        self.input_height = input_height
        self.input_channel_count = input_channel_count
        self.output_channel_count = output_channel_count

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=self.input_channel_count, out_channels=16, kernel_size=kernel_size, stride = stride, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride =2, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=7),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=self.output_channel_count, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):

        # Get latent representation
        latent = self.encoder(x)

        # Reconstruct input
        reconstructed = self.decoder(latent)

        return reconstructed

In [9]:
model = AutoEncoder(input_channel_count=1, output_channel_count=1).to(device)

In [10]:
# Loss function and optimizer
criterion = nn.L1Loss()
# criterion = nn.KLDivLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [11]:
def train(debug=True, num_epochs=1, max_samples=10, print_per=20, batch_size=16):
    """
    Main Training loop Function
    """
    sample_ix = 0
    
    for epoch in range(num_epochs):
        total_loss = 0
        epoch_samples = 0
        ipipe = iter(fullsat)  # Make an iterator to walk the time period

        batch = []
        start_time = time.time()
        while True:
            try:
                sample = next(ipipe)
            except StopIteration:
                break  # advance the epoch loop
            except:
                pass # some samples are just missing

            # If the collected sample contains any nans, continue to next iteration to pick up another sample
            if torch.isnan(torch.tensor(sample)).any():
                continue

            batch.append(sample[0]) # Drops the batch size dimension (it's 1)
            sample_ix += 1
            epoch_samples += 1

            # Currently will not process remaining portion of final batch
            if sample_ix > max_samples:
                break

            # If the batch does not contain enough items yet
            if len(batch) < batch_size:
                continue

            x = torch.from_numpy(np.stack(batch)).float().to(device)
    
            optimizer.zero_grad()
    
            # Forward pass
            y = model.forward(x)
            loss = criterion(y, x)
    
            # Backward pass and optimize        
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()

            batch = []

            if epoch_samples % print_per == 0:
                print(time.time() - start_time)
                print(f"[Epoch {epoch+1}] Sample {epoch_samples}, Batch Loss: {loss.item():.4f}")
                start_time = time.time()
    
        # Print epoch statistics
        avg_loss = total_loss / epoch_samples
        epoch_samples = 0  # Reset for next epoch
        print(f'Epoch [{epoch+1}/{epoch_samples}], Average Loss: {avg_loss:.4f}')

In [12]:
%%time
train(debug=False, num_epochs=1, max_samples=5000, print_per = 128, batch_size=32)

-25.83867049217224
[Epoch 1] Sample 128, Loss: 0.2385
-24.91391658782959
[Epoch 1] Sample 256, Loss: 0.2473
Epoch [1/0], Average Loss: 0.0075
CPU times: user 2min 1s, sys: 5.92 s, total: 2min 7s
Wall time: 54.6 s


In [13]:
fullsat_validate = petpipe.Pipeline(
    satpipe,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20210101T00', '20220101T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

In [ ]:
training_iterator = iter(fullsat) 
sample_numpy = next(training_iterator)

validate_iterator = iter(fullsat_validate)

def next_sample_gpu(pet_iterator):
    return torch.from_numpy(next(pet_iterator)).float().to(device)

sample_tensor_gpu = torch.from_numpy(sample_numpy).float().to(device)
sample_prediction_gpu = model.forward(sample_tensor_gpu)
sample_prediction_gpu = model.forward(next_sample_gpu(training_iterator))

def prediction_from_iterator(pet_iterator, model):
    sample_numpy = next(pet_iterator)
    sample_gpu = torch.from_numpy(sample_numpy).float().to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu = sample_prediction_gpu.to('cpu').detach().numpy()
    return prediction_cpu

prediction_from_iterator(validate_iterator, model)

In [ ]:
fig1 = plt.figure('sidebyside_satellite', figsize=(24,8))
for ix1 in range(3):
    sample_numpy = next(validate_iterator)
    sample_gpu = torch.from_numpy(sample_numpy).float().to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu = prediction_gpu.to('cpu').detach().numpy()
    ax1 = fig1.add_subplot(3,3,(ix1*3)+1)
    ax1.imshow(sample_numpy[0,0])
    ax1 = fig1.add_subplot(3,3,(ix1*3)+2)
    ax1.imshow(prediction_cpu[0,0])
    ax1 = fig1.add_subplot(3,3,(ix1*3)+3)
    ax1.imshow(prediction_cpu[0,0] - sample_numpy[0,0])  

In [ ]:
import torchmetrics.image
ssi_metric = torchmetrics.image.StructuralSimilarityIndexMeasure()
rmse_sw_metric = torchmetrics.image.RootMeanSquaredErrorUsingSlidingWindow()

ssi_values = []
rmse_values = []
for ix1 in range(10):
    sample_numpy = next(validate_iterator)
    sample_cpu = torch.from_numpy(sample_numpy).float()
    sample_gpu = sample_cpu.to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu_tensor = prediction_gpu.to('cpu').detach()
    
    ssi_values += [float(ssi_metric(sample_cpu,  prediction_cpu_tensor)) ]
    rmse_values += [float(rmse_sw_metric(sample_cpu,  prediction_cpu_tensor))]

def mean(list_):
    print(type(list_))
    return sum(list_)/len(list_)

print(f"Mean SSE {mean(ssi_values)}")
print(f"Mean RMSE {mean(rmse_values)}")